# GraphRAG Query Modes Test

This notebook demonstrates how to build local search contexts and perform retrieval using GraphRAG components.

Added section headers and inline comments to make the flow clearer.

In [ ]:
# 0. Imports and setup
# Core libraries and GraphRAG components used in examples below
import pandas as pd
from pathlib import Path
from graphrag.query.context_builder.entity_extraction import EntityVectorStoreKey
from graphrag.query.structured_search.local_search.mixed_context import LocalSearchMixedContext
from graphrag_vectors.lancedb import LanceDBVectorStore

In [2]:
from graphrag.query.indexer_adapters import (
    read_indexer_covariates,
    read_indexer_entities,
    read_indexer_relationships,
    read_indexer_reports,
    read_indexer_text_units,
)

In [ ]:
# 1. Load your indexed data (parquet files from your output folder)
# Update the path below if your `output` directory is located elsewhere
root = Path("./output")
# Example reads (uncomment to run if parquet files exist):
# entity_df = pd.read_parquet(root / "entities.parquet")
# relationships = pd.read_parquet(root / "relationships.parquet")
# community_df = pd.read_parquet(root / "community_reports.parquet")
# text_units = pd.read_parquet(root / "text_units.parquet")

In [ ]:
# Paths and table names used by the indexer and vector store
OUTPUT_DIR = "./output"
LANCEDB_URI = f"{OUTPUT_DIR}/lancedb"

COMMUNITY_REPORT_TABLE = "community_reports"
ENTITY_TABLE = "entities"
COMMUNITY_TABLE = "communities"
RELATIONSHIP_TABLE = "relationships"
COVARIATE_TABLE = "covariates"
TEXT_UNIT_TABLE = "text_units"
COMMUNITY_LEVEL = 2

In [ ]:
# Read entity and community tables (used for mapping and ranking)
entity_df = pd.read_parquet(f"{OUTPUT_DIR}/{ENTITY_TABLE}.parquet")
community_df = pd.read_parquet(f"{OUTPUT_DIR}/{COMMUNITY_TABLE}.parquet")

# Convert raw tables into the internal indexer structures
entities = read_indexer_entities(entity_df, community_df, COMMUNITY_LEVEL)
print(f"Entity count: {len(entity_df)}")
entity_df.head()

Entity count: 201


,id,human_readable_id,title,type,description,text_unit_ids,frequency,degree
0,54c5690a-ef57-43d7-a625-de03dadba989,0,DECEMBER,NOUN PHRASE,,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...,2,30
1,c482e454-f621-447d-ba64-7fbbb7123314,1,CHRISTMAS PRESENT,NOUN PHRASE,,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...,6,17
2,ebd02c68-b678-4f27-9118-fec281c4e782,2,MAY,NOUN PHRASE,,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...,6,19
3,0ec8f093-4cf5-47f3-858c-83a11d2d9886,3,THE FIRST OF THE THREE SPIRITS,NOUN PHRASE,,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...,2,33
4,34b1c878-ada0-4f8e-9d2c-607a8152b612,4,RESTRICTIONS WHATSOEVER,NOUN PHRASE,,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...,3,35


In [ ]:
# Read relationships (edges) used to provide structured local context
relationship_df = pd.read_parquet(f"{OUTPUT_DIR}/{RELATIONSHIP_TABLE}.parquet")
relationships = read_indexer_relationships(relationship_df)

print(f"Relationship count: {len(relationship_df)}")
relationship_df.head()

Relationship count: 2105


,id,human_readable_id,source,target,description,weight,combined_degree,text_unit_ids
0,03535beb-5131-410c-abc4-683a467848b7,0,BOB CRATCHIT,CAROLINE,,0.000201,65,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...
1,9c5578bf-95df-41c0-956e-215dcba5c927,1,BOB CRATCHIT,CHRISTMAS,,0.000281,64,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...
2,bd3c51be-2fb2-455e-b42c-017888f065e2,2,BOB CRATCHIT,CHRISTMAS CAROL,,0.000163,71,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...
3,1269b96b-14c2-46a2-9b9a-95f019d9a3db,3,BOB CRATCHIT,CHRISTMAS PRESENT,,0.000327,53,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...
4,12597e32-c94c-4240-b7b9-6f41dfca9e3f,4,BOB CRATCHIT,FRED,,0.000117,53,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...


In [ ]:
# Read community reports (long-form context sources)
report_df = pd.read_parquet(f"{OUTPUT_DIR}/{COMMUNITY_REPORT_TABLE}.parquet")
reports = read_indexer_reports(report_df, community_df, COMMUNITY_LEVEL)

print(f"Report records: {len(report_df)}")
report_df.head()

Report records: 4


,id,human_readable_id,community,level,parent,children,title,summary,full_content,rank,rating_explanation,findings,full_content_json,period,size
0,4f63a992d87729a065216f9e06f6f3256921ab62646afb...,33,33,2,12,[],A Christmas Carol Character Analysis as of 1843,This report delves into the key characters of ...,# A Christmas Carol Character Analysis as of 1...,8.5,The moderate to high importance rating reflect...,"[{'explanation': 'The protagonist, Ebenezer Sc...","{\n ""title"": ""A Christmas Carol Character A...",2026-06-21,3
1,75d01e439200ef1f12848511c60e280e0e3a733783cbd2...,46,46,2,19,[],The Isolated World of Ebenezer Scrooge,This report provides an overview of Ebenezer S...,# The Isolated World of Ebenezer Scrooge\n\nTh...,8.5,The importance rating reflects the significant...,[{'explanation': 'Ebenezer Scrooge is portraye...,"{\n ""title"": ""The Isolated World of Ebeneze...",2026-06-21,5
2,d922ffc9c5967c272742566bcd6e33e0b90041d35cf323...,47,47,2,24,[],A Christmas Carol: Scrooge's Encounter with Ma...,The report delves into Ebenezer Scrooge's enco...,# A Christmas Carol: Scrooge's Encounter with ...,8.5,The high importance rating reflects the pivota...,[{'explanation': 'The encounter between Scroog...,"{\n ""title"": ""A Christmas Carol: Scrooge's ...",2026-06-21,3
3,23dbf4ced82b19567224c0f3f56ec184fce9ae26ba827b...,9,9,1,0,[],Project Gutenberg Literary Archive Foundation ...,This report provides an overview of the Projec...,# Project Gutenberg Literary Archive Foundatio...,8.5,The high importance rating reflects the critic...,[{'explanation': 'The Project Gutenberg Litera...,"{\n ""title"": ""Project Gutenberg Literary Ar...",2026-06-21,11


In [ ]:
# Read smaller text units (used for direct retrieval and summarization)
text_unit_df = pd.read_parquet(f"{OUTPUT_DIR}/{TEXT_UNIT_TABLE}.parquet")
text_units = read_indexer_text_units(text_unit_df)

print(f"Text unit records: {len(text_unit_df)}")
text_unit_df.head()

Text unit records: 42


,id,human_readable_id,text,n_tokens,document_id,entity_ids,relationship_ids,covariate_ids
0,9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb652...,0,﻿The Project Gutenberg eBook of A Christmas Ca...,1200,aac6608d94fe68e17f3b7c6e7f266528e9d9bcef244ee8...,"[54c5690a-ef57-43d7-a625-de03dadba989, c482e45...","[03535beb-5131-410c-abc4-683a467848b7, 9c5578b...",[]
1,aec1597166b241ac475086aa7e94826f5eebd1b9659507...,1,"The air was filled with phantoms,\n wanderi...",1200,aac6608d94fe68e17f3b7c6e7f266528e9d9bcef244ee8...,"[6ac62b41-e242-419d-be2c-f094ac535d1d, 4f9ae05...","[21f02a0a-26c4-458b-a81d-c7f1ec4bbfd5, e252eb7...",[]
2,332deb9b6b27ca253056816eb64198676c9ba75f952fe7...,2,", Scrooge! a\nsqueezing, wrenching, grasping, ...",1200,aac6608d94fe68e17f3b7c6e7f266528e9d9bcef244ee8...,"[d3117b13-426b-4d5f-9a6d-7509e1712ab8, 63de212...","[0171acdb-561c-4c8d-a387-89e02d6806c6, 4fff723...",[]
3,4d64d0ec8e5ab02f22ce3da6c5dba0d27690582293792b...,3,", uncle!' said the nephew.\n\n'What else can I...",1200,aac6608d94fe68e17f3b7c6e7f266528e9d9bcef244ee8...,"[a58ab232-d565-4500-80d5-f3613d8e8400, d3117b1...","[0171acdb-561c-4c8d-a387-89e02d6806c6, 4fff723...",[]
4,8aa9443f89e89b6e1fc489b720acd4a21c77f00cc806bb...,4,": THEY WERE PORTLY GENTLEMEN, PLEASANT TO BEHO...",1200,aac6608d94fe68e17f3b7c6e7f266528e9d9bcef244ee8...,"[ebd02c68-b678-4f27-9118-fec281c4e782, d3117b1...","[01a45293-1362-46e2-b418-38de11dedf06, 77f6401...",[]


In [ ]:
# Initialize Vector Store (LanceDB) containing entity embeddings
vector_store = LanceDBVectorStore(
    db_uri=LANCEDB_URI,
    index_name="entity_description",  # GraphRAG stores entity embeddings in this table
)
vector_store.connect()
print(f"LanceDB connected: {vector_store.document_collection is not None}")
print(f"Entity embedding rows: {vector_store.count()}")

LanceDB connected: True
Entity embedding rows: 201


In [ ]:
# Embedding model configuration (example). Replace with your provider details.
from graphrag_llm.config.model_config import ModelConfig
from graphrag_llm.config.types import LLMProviderType

embedding_config = ModelConfig(
    type=LLMProviderType.LiteLLM,         # Recommended: use the Enum rather than a raw string
    model_provider="openai",              # REQUIRED: specify provider (e.g., 'openai')
    model="mxbai-embed",
    api_key="sk-1234567890",
    api_base="http://localhost:4000/v1",
    model_id="default_embedding_model"
)

In [ ]:
# Instantiate the embedding helper using the configuration above
from graphrag_llm.embedding import LLMEmbedding, create_embedding
from graphrag_llm.types import LLMEmbeddingResponse
llm_embedding: LLMEmbedding = create_embedding(embedding_config)

In [ ]:
# Quick embedding smoke-test: ensures the embedder returns vectors
embeddings_batch: LLMEmbeddingResponse = llm_embedding.embedding(
    input=["Hello world", "How are you?"]
)
for embedding in embeddings_batch.embeddings:
    print(embedding[0:3])

[0.024303264915943146, 0.0112022515386343, 0.014001327566802502]
[0.05620148777961731, -0.02417628839612007, -0.06722883880138397]


In [ ]:
# Initialize the context builder that will produce retrieval contexts
context_builder = LocalSearchMixedContext(
    community_reports=reports,
    text_units=text_units,
    entities=entities,
    relationships=relationships,
    entity_text_embeddings=vector_store,
    embedding_vectorstore_key=EntityVectorStoreKey.ID,
    text_embedder=llm_embedding, 
    # token_encoder=your_token_encoder,   # Provide a token encoder to accurately size context windows
)

In [ ]:
# Example retrieval: returns context records and chunks for the query
query = "How do the three spirits change Scrooge's behavior?"
context_result = context_builder.build_context(
    query=query,
    max_context_tokens = 8000, 
    text_unit_prop = 0.5, # Token % allocated to text unit
    community_prop = 0.25, # Token % allocated to community reports
    # Token % allocated to local data (entities/relationships) = 1 - text_unit_prop - community_prop
    top_k_mapped_entities = 2, 
    top_k_relationships = 2,
    use_community_summary = True, # Use shorter summaries instead of full reports
    min_community_rank=1, # Filter out community reports that don't have enough connections.
)

In [32]:
context_result.context_records

{'reports':    id                                              title  \
 0  33    A Christmas Carol Character Analysis as of 1843   
 1  47  A Christmas Carol: Scrooge's Encounter with Ma...   
 
                                              summary  
 0  This report delves into the key characters of ...  
 1  The report delves into Ebenezer Scrooge's enco...  ,
 'relationships':     id            source                          target description links  \
 0   86   CHRISTMAS CAROL  THE FIRST OF THE THREE SPIRITS                 2   
 1  165  EBENEZER SCROOGE  THE FIRST OF THE THREE SPIRITS                 2   
 2   75   CHRISTMAS CAROL                    ILLUSTRATION                 3   
 3  243      ILLUSTRATION  THE FIRST OF THE THREE SPIRITS                 3   
 4   76   CHRISTMAS CAROL                           LADEN                 3   
 5  284             LADEN  THE FIRST OF THE THREE SPIRITS                 3   
 6   82   CHRISTMAS CAROL                      STAVE FIVE        

In [33]:
print(context_result.context_chunks)

-----Reports-----
id|title|summary
33|A Christmas Carol Character Analysis as of 1843|This report delves into the key characters of Charles Dickens' A Christmas Carol, illustrating how these entities interact within the narrative framework to convey themes of redemption and personal growth. The information is relevant to the story's climax around Christmas time in the 1840s.
47|A Christmas Carol: Scrooge's Encounter with Marley's Ghost|The report delves into Ebenezer Scrooge's encounter with Marley's Ghost, exploring how the entities interact within the narrative framework to convey the themes of redemption and the supernatural. The information is relevant to the events that unfold in the story as Scrooge is warned about his fate and the visits from the Three Spirits.


-----Entities-----
id|entity|description
137|UNCLE SCROOGE|
22|EBENEZER SCROOGE|
15|CHRISTMAS CAROL|
3|THE FIRST OF THE THREE SPIRITS|


-----Relationships-----
id|source|target|description|links
86|CHRISTMAS CAROL|THE 

In [37]:
from utilities.graph_rag_context import graphrag_retriever

chunks = graphrag_retriever(
    "How do the three spirits change Scrooge's behavior?",
    # max_context_tokens=4000,
    # text_unit_prop=0.5,
    # community_prop=0.25,
    # top_k_mapped_entities=1,
    # top_k_relationships=1,
    # use_community_summary=True,
    # min_community_rank=1,
)

Reached token limit - reverting to previous context state


In [38]:
print(chunks)

-----Reports-----
id|title|content
33|A Christmas Carol Character Analysis as of 1843|"# A Christmas Carol Character Analysis as of 1843

This report delves into the key characters of Charles Dickens' A Christmas Carol, illustrating how these entities interact within the narrative framework to convey themes of redemption and personal growth. The information is relevant to the story's climax around Christmas time in the 1840s.

## Scrooge's Transformation

The protagonist, Ebenezer Scrooge, undergoes a pivotal transformation after being visited by the Ghosts of Christmas Past, Present, and Yet to Come. This transformation is highlighted through his interactions with the spectral beings, showcasing his growth from a miserly, cold-hearted individual to one who is willing to change and honor Christmas in his heart. Effective character development not only prevents Scrooge's further descent into miserliness but also supports the story's themes of redemption and the importance of kindness. [